In [ ]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score
import sys
import os

from data_pipeline import get_full_pipeline

RF_CONFIG = {
    "base": dict(n_estimators=100, max_depth=None, max_features="sqrt", min_samples_leaf=1),
    "small": dict(n_estimators=50, max_depth=12, max_features="sqrt", min_samples_leaf=2),
    "tiny": dict(n_estimators=20, max_depth=8, max_features="sqrt", min_samples_leaf=4)
}


def run_rf_baseline(mode: int, size="base", batch_size=2048):
  cfg = RF_CONFIG[size]

  (loaders, scaler, le) = get_full_pipeline(mode, batch_size=batch_size)

  train_loader, val_loader, test_loader, n_features, n_classes = loaders

  model = RandomForestClassifier(
      **cfg,
      n_jobs=-1,
      random_state=42
  )

  # make dataset tensors and convert to numpy
  X_train, y_train = train_loader.dataset.tensors
  X_val, y_val = val_loader.dataset.tensors
  X_test, y_test = test_loader.dataset.tensors

  X_train = X_train.numpy()
  y_train = y_train.numpy()
  X_val = X_val.numpy()
  y_val = y_val.numpy()
  X_test = X_test.numpy()
  y_test = y_test.numpy()

  model.fit(X_train, y_train)

  test_preds = model.predict(X_test)

  accuracy = accuracy_score(y_test, test_preds)
  precision, recall, f1, _ = precision_recall_fscore_support(y_test, test_preds, average='macro', zero_division=0)

  cm = confusion_matrix(y_test, test_preds)

  train_losses = [1 - accuracy]
  val_losses = [1 - accuracy]

  return {
      'accuracy': accuracy,
      'precision': precision,
      'recall': recall,
      'f1': f1,
      'train_loss': train_losses,
      'val_loss': val_losses,
      'confusion_matrix': cm,
      'history': val_losses
  }